In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita visualizacao grafica inline no Jupyter notebook
%matplotlib inline

# Da gravacao de EEG ao DataLoader do PyTorch

**Dificuldade 1** | **Tempo de execucao: 2m** | **Computacao: CPU**

Um modelo treina em lotes de tensores, nao em tracados continuos de voltagem. Este tutorial fecha essa lacuna em uma gravacao BIDS do [OpenNeuro](https://openneuro.org) ``ds002718`` :cite:`wakeman2015`, acessivel atraves do [NEMAR](https://nemar.org) :cite:`delorme2022nemar`: dois pre-processadores seguros, uma etapa de janelamento de comprimento fixo, um :class:`DataLoader <torch.utils.data.DataLoader>` :cite:`paszke2019pytorch` e um cache Zarr opcional que transforma leituras de lotes em acessos aleatorios de poucos milissegundos. Nao treinamos um modelo aqui. A entrega final e o formato (``shape``) e o tipo de dados (``dtype``) de um lote.

.. sphinx_gallery_thumbnail_path = '_static/thumbs/plot_02_dataset_to_dataloader.png'
Palavras-chave: carregamento, PyTorch, DataLoader


## Objetivos de aprendizagem
- Encadeie :class:`~eegdash.api.EEGDashDataset`, :func:`braindecode.preprocessing.preprocess` e :func:`~braindecode.preprocessing.create_fixed_length_windows` em um unico sujeito.
- Preveja o formato de uma janela a partir de ``(n_channels, window_seconds * sfreq)`` e verifique o resultado.
- Leia um lote do :class:`torch.utils.data.DataLoader` e compreenda o significado de cada eixo.
- Converta o conjunto de dados janelado em um armazenamento Zarr e faca a releitura para obter velocidade de acesso aleatorio.
- Escolha valores seguros para ``batch_size``, ``num_workers``, ``pin_memory`` e ``shuffle`` para cargas de trabalho de EEG.



## Requisitos
- Cerca de 2 min em CPU na primeira execucao; menos de 20 s uma vez em cache.
- Rede na primeira chamada (~30-60 MB gravados em ``cache_dir``); offline posteriormente.
- Pre-requisitos: :doc:`plot_00_first_search` (catalogo), :doc:`plot_01_first_recording` (um objeto ``Raw``).
- Conceito: :doc:`/concepts/eegdash_objects`.



Configuracao inicial. ``np.random.seed`` e ``torch.manual_seed`` tornam ``shuffle=True`` e qualquer inicializacao de modelo reproduziveis entre execucoes (E3.21).



In [ ]:
# Importa utilitarios do sistema operacional e manipulacao de diretorios
import os
from pathlib import Path

# Importa bibliotecas para plotagem, computacao matricial e manipulacao de tabelas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Importa PyTorch e a classe DataLoader para iteracao em lotes
import torch
from torch.utils.data import DataLoader

# Importa modulos do Braindecode e do EEGDash para pre-processamento e janelamento
import braindecode
import eegdash
from braindecode.preprocessing import (
    Preprocessor,
    create_fixed_length_windows,
    preprocess,
)
from eegdash import EEGDashDataset
from eegdash.viz import use_eegdash_style

# Aplica tema visual do EEGDash
use_eegdash_style()
# Define semente pseudoaleatoria para reprodutibilidade no NumPy e PyTorch
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Garante a criacao do diretorio de cache persistente
CACHE_DIR = Path(os.environ.get("EEGDASH_CACHE_DIR", Path.home() / ".eegdash_cache"))
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(
    f"eegdash {eegdash.__version__}; braindecode {braindecode.__version__}; "
    f"torch {torch.__version__}"
)
print(f"cache_dir={CACHE_DIR}")

## Dataset vs DataLoader: o modelo mental
O tutorial oficial do PyTorch estabelece a distincao com clareza: um :class:`~torch.utils.data.Dataset` *gerencia o armazenamento e recuperacao das amostras* (``__len__`` e ``__getitem__``); um :class:`~torch.utils.data.DataLoader` *consome* o dataset e acrescenta agrupamento em lotes (*batching*), embaralhamento (*shuffling*) e orquestracao de processos trabalhadores (*workers*). O DataLoader nao e uma estrutura de armazenamento de dados. Ele e um iteravel que invoca ``__getitem__`` no dataset subjacente e empilha os resultados.

No contexto do EEG, o fluxo envolve tres formatos que voce pode manter em mente:

```text
EEGDashDataset                  WindowsDataset                DataLoader
(registros + metadados BIDS)    (amostras cortadas p/ modelo) (consumidor)
┌────────────────────┐          ┌───────────────────┐         ┌────────────┐
│ registro 0 (Raw) ─┐│ pre-proc │ janela 0 (X, y)   │ lote +  │ lote 0     │
│ registro 1 (Raw) ─┼┼─────────▶│ janela 1 (X, y)   │ embaral.│ lote 1     │
│ registro 2 (Raw) ─┘│ + corte  │ ...               │────────▶│ ...        │
│ ...                │ janelas  │ janela N (X, y)   │         │ lote K     │
└────────────────────┘          └───────────────────┘         └────────────┘
  __len__  = n_registros         __len__  = n_janelas           iter() gera
  __getitem__ -> (raw, ...)       __getitem__ -> (X, y, idx)      tensores empilhados
```
Apos a execucao do pipeline, revisitaremos esses formatos com as dimensoes reais produzidas em tempo de execucao.



## Cinco corolarios

- **Uma amostra e uma janela, nao uma gravacao completa.** O EEG continuo e uma matriz longa por sessao. Modelos de aprendizado profundo treinam em quadros de comprimento fixo, portanto o pipeline corta cada ``Raw`` em tensores de dimensao ``(n_channels, window_samples)`` antes que qualquer agrupamento em lotes aconteca.
- **Composicao do pipeline.** ``EEGDashDataset`` (registros e metadados BIDS) alimenta :func:`~braindecode.preprocessing.preprocess` (edicoes in-place em cada :class:`mne.io.Raw`), que alimenta :func:`~braindecode.preprocessing.create_fixed_length_windows` (continuo para janelado), que alimenta :class:`~torch.utils.data.DataLoader` (janelado para lotes). Cada estagio propaga os metadados BIDS, garantindo que divisoes posteriores preservem a identidade do sujeito (veja :doc:`/generated/auto_examples/tutorials/10_core_workflow/plot_11_leakage_safe_split`).
- **Duas abordagens de janelamento.** :func:`~braindecode.preprocessing.create_fixed_length_windows` avanca com passo constante ao longo do sinal continuo, ignorando eventos; adequado para pre-treinamento autosupervisionado e estagiamento de sono. Por outro lado, :func:`~braindecode.preprocessing.create_windows_from_events` corta o sinal ao redor dos marcadores de eventos BIDS com deslocamentos explicitos; e a escolha ideal para ERPs e tarefas associadas a eventos (faces/embaralhadas, oddball, imaginacao motora). Ambos retornam um :class:`~braindecode.datasets.BaseConcatDataset` de :class:`~braindecode.datasets.WindowsDataset` e integram-se diretamente ao mesmo DataLoader sem alteracao de codigo.
- **Acesso aleatorio vs. armazenamento sequencial.** O treinamento embaralha janelas; o armazenamento de sinal subjacente precisa viabilizar consultas ``X[i]`` de baixo custo. O formato ``.fif`` funciona bem para uma unica gravacao, mas seu custo de busca cresce linearmente com o tamanho do arquivo; o Zarr armazena blocos de tamanho fixo e le qualquer janela em dezenas de milissegundos (veja a Etapa 5).
- **Carregamento preguicoso (*lazy*) por padrao.** A instanciacao do dataset consulta apenas o indice de metadados. O objeto ``Raw`` do MNE e materializado sob demanda (``record.raw``). O parametro ``preload=True`` nas funcoes de janelamento forca o sinal cortado para a memoria RAM, o que representa a escolha ideal quando a gravacao cabe na memoria.
- **Conexoes anteriores e posteriores.** O tutorial ``plot_01`` abriu este mesmo registro e inspecionou seu espectro. O tutorial ``plot_10`` aborda a receita completa de pre-processamento (montagem, referencia, filtro passa-banda) que aqui mantemos minima. O tutorial ``plot_11`` realiza divisoes sem vazamento de dados. O tutorial ``plot_13`` salva e recarrega janelas preparadas para evitar reprocessamentos em sessoes posteriores.



## O que um conjunto de dados janelado pode fazer?
Antes de construir um, liste os metodos que o :class:`braindecode.datasets.WindowsDataset` expoe; a maioria deles sao os verbos dos quais o DataLoader depende implicitamente (``__len__``, ``__getitem__``, ``set_description``, ``transform``).



In [ ]:
# Importa a classe WindowsDataset da Braindecode
from braindecode.datasets import WindowsDataset

# Identifica metodos publicos disponiveis na classe de janelas
windows_methods = sorted(
    name
    for name in dir(WindowsDataset)
    if not name.startswith("_") and callable(getattr(WindowsDataset, name, None))
)
# Exibe os metodos identificados em formato tabular
pd.DataFrame({"method": windows_methods}).head(20)

## Etapa 1: Construir o dataset (carregamento preguicoso / lazy)
Mesmo padrao do tutorial ``plot_01``. O uso de um unico sujeito mantem a execucao dentro do tempo didatico; os filtros da linguagem BIDS sao aplicados diretamente :cite:`pernet2019eegbids`.



In [ ]:
# Define parametros BIDS para selecao de um sujeito e tarefa especificos
DATASET = "ds002718"
SUBJECT = "002"  # Principio de minimalidade: um sujeito, uma tarefa
TASK = "FaceRecognition"
# Instancia o dataset
dataset = EEGDashDataset(
    cache_dir=CACHE_DIR, dataset=DATASET, subject=SUBJECT, task=TASK
)
# Acessa a primeira gravacao e seu objeto Raw subjacente
record = dataset.datasets[0]
raw = record.raw
# Exibe resumo das caracteristicas basicas da gravacao
pd.Series(
    {
        "n_recordings": len(dataset.datasets),
        "n_channels": raw.info["nchan"],
        "sfreq (Hz)": float(raw.info["sfreq"]),
        "duration (s)": round(raw.times[-1], 1),
    },
    name="value",
).to_frame()

## Etapa 2: Dois pre-processadores seguros
**Preveja.** Como ficara ``raw.info['sfreq']`` apos uma reamostragem para 100 Hz? E ``len(raw.ch_names)`` apos descartar canais que nao sejam de EEG?

**Execute.** ``pick_types(eeg=True)`` reteem apenas sensores de EEG, e ``resample(sfreq=100)`` reduz a taxa de amostragem. Filtragem e referencia detalhadas sao tratadas no tutorial ``plot_10``; aqui mantemos a receita restrita a duas etapas nomeadas para focar na transicao ``Raw -> janelas -> DataLoader``.



In [ ]:
# Define taxa de amostragem alvo em Hz
TARGET_SFREQ = 100  # Hz
# Aplica pipeline de pre-processamento com selecao estrita de canais EEG e reamostragem
preprocess(
    dataset,
    [
        Preprocessor("pick_types", eeg=True, eog=False, misc=False),
        Preprocessor("resample", sfreq=TARGET_SFREQ),
    ],
)
# Inspeciona o objeto Raw apos o pre-processamento
raw_pp = dataset.datasets[0].raw
n_channels = len(raw_pp.ch_names)
sfreq = float(raw_pp.info["sfreq"])
# Exibe contagem de canais, nova frequencia de amostragem e tipo de dado do array
pd.Series(
    {
        "n_channels": n_channels,
        "sfreq (Hz)": sfreq,
        "dtype": str(raw_pp.get_data().dtype),
    },
    name="value",
).to_frame()

## Etapa 3: Fatiar em janelas de comprimento fixo
Dois parametros determinam todas as dimensoes a partir deste ponto: tamanho da janela e passo (*stride*).

- ``window_size_samples = window_seconds * sfreq``.
- ``window_stride_samples = window_size_samples`` resulta em 0% de sobreposicao; cada amostra do sinal continuo entra em exatamente uma janela.
- Reduzir o passo pela metade dobra o numero de janelas e introduz correlacao entre quadros adjacentes de uma mesma gravacao, o que prejudica a avaliacao se as divisoes de validacao nao levarem em conta a identidade do participante (divisoes inter-sujeito em ``plot_11``).

**Preveja.** Qual sera o valor de ``len(windows)`` para uma gravacao de 1 segundo a 100 Hz com janelas de 2 segundos? (Resposta: zero, devido ao descarte de janelas incompletas com ``drop_last_window=True``.)



In [ ]:
# Define comprimento da janela em segundos
WINDOW_SECONDS = 2.0
# Converte duracao para numero de amostras com base na taxa de amostragem alvo (2.0s * 100 Hz = 200 amostras)
window_size_samples = int(WINDOW_SECONDS * TARGET_SFREQ)
# Cria as janelas de comprimento fixo com 0% de sobreposicao e pre-carregamento em memoria RAM
windows = create_fixed_length_windows(
    dataset,
    window_size_samples=window_size_samples,
    window_stride_samples=window_size_samples,  # 0% de sobreposicao
    drop_last_window=True,
    preload=True,
)
# Extrai a primeira janela como exemplo para validacao dimensional
X_one, y_one, _idx = windows[0]
# Exibe resumo da estrutura dimensional gerada
pd.Series(
    {
        "n_windows": len(windows),
        "windows[0][0].shape": str(tuple(X_one.shape)),
        "X.dtype": str(X_one.dtype),
        "window_samples": window_size_samples,
        "window_seconds": WINDOW_SECONDS,
    },
    name="value",
).to_frame()

## Etapa 4: Encapsular em um DataLoader
Quatro parametros principais sao relevantes para cargas de trabalho de EEG; os demais argumentos do ``DataLoader`` podem ser mantidos nos valores padrao ate a etapa de treinamento efetivo do modelo.

- ``batch_size``. De 8 a 32 e um intervalo inicial seguro para testes e depuracao em CPU; valores finais dependem da memoria da GPU.
- ``shuffle``. ``True`` para treinamento, ``False`` para avaliacao. Quando ``True``, requer um :class:`torch.Generator` com semente fixa se a reprodutibilidade da ordem do primeiro lote for necessaria.
- ``num_workers``. ``0`` (sincrono) e o padrao correto ao usar ``preload=True``, pois as janelas ja residem na memoria RAM. Valores ``>0`` auxiliam somente quando o dataset precisa ler arquivos do disco em cada chamada de ``__getitem__`` (como no caso do Zarr abaixo).
- ``pin_memory``. Defina como ``True`` caso um dispositivo CUDA esteja disponivel e voce planeje transferir os lotes com ``.to(device, non_blocking=True)``.



In [ ]:
# Define tamanho do lote (batch)
BATCH_SIZE = 8
# Instancia gerador do PyTorch com semente fixa para reprodutibilidade no embaralhamento
gen = torch.Generator().manual_seed(SEED)
# Cria o DataLoader do PyTorch
loader = DataLoader(
    windows,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    generator=gen,
)
# Recupera o primeiro lote gerado pelo iterador
X_batch, y_batch, _idx_batch = next(iter(loader))
# Exibe propriedades do lote gerado
pd.Series(
    {
        "X.shape": str(tuple(X_batch.shape)),
        "X.dtype": str(X_batch.dtype),
        "y.shape": str(tuple(y_batch.shape)),
        "y unique": str(torch.unique(y_batch).tolist()),
        "pin_memory": loader.pin_memory,
        "num_workers": loader.num_workers,
    },
    name="value",
).to_frame()

**Investigue.** O tensor do lote possui o formato ``(batch_size, n_channels, window_size_samples)`` com tipo de ponto flutuante (float32). Essa e exatamente a dimensao consumida por modelos do Braindecode, como :class:`~braindecode.models.ShallowFBCSPNet` e :class:`~braindecode.models.EEGNetv4`.



## Visao geral do pipeline: os formatos reais
Agora que cada etapa foi executada, podemos revisitar o diagrama conceitual com os numeros reais produzidos pelo ambiente.



In [ ]:
# Bloco opcional para geracao de diagramas de pipeline com biblioteca auxiliar
# from _pipeline_diagram import draw_pipeline

# fig_pipe = draw_pipeline(
#     record_signal=raw_pp.get_data(picks="eeg").copy(),
#     window_xy=np.asarray(windows[0][0]).copy(),
#     batch_xy=np.asarray(X_batch[0]).copy(),
#     n_records=len(windows.datasets),
#     n_channels=n_channels,
#     sfreq=sfreq,
#     window_size_samples=window_size_samples,
#     batch_size=BATCH_SIZE,
#     n_windows=len(windows),
#     subject=SUBJECT,
#     n_channels_full=raw_pp.info["nchan"],
# )
# plt.show()

## Reprodutibilidade: qual fonte aleatoria seleciona qual janela?
Duas sementes e um ``Generator`` cobrem os principais modos de falha que afetam pipelines de EEG.

- ``np.random.seed`` e ``torch.manual_seed`` tornam o processo *principal* deterministico: inicializacao do modelo, ordenacao base quando ``num_workers=0`` e chamadas ao NumPy dentro do metodo ``__getitem__`` do dataset.
- Um :class:`torch.Generator` fornecido ao DataLoader fixa a *ordem de amostragem*. Sem ele, duas execucoes com ``shuffle=True`` podem produzir janelas distintas no primeiro lote, pois o gerador global do PyTorch pode ter avancado devido a outras operacoes.
- Com ``num_workers > 0``, cada processo filho bifurca o estado do processo pai. O NumPy nao reinicia suas sementes automaticamente (enquanto o PyTorch o faz com sementes deterministicas por trabalhador). O padrao seguro e a funcao ``seed_worker(worker_id)`` demonstrada abaixo, conforme indicado nas diretrizes oficiais de reprodutibilidade do PyTorch.



In [ ]:
# Importa o modulo random padrao da biblioteca Python
import random


# Funcao para redefinir sementes de bibliotecas terceiras em cada worker secundario
def seed_worker(worker_id):
    """Redefine sementes do numpy e do modulo random padrao em cada processo worker do DataLoader."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# Configura DataLoader com worker_init_fn e Generator deterministico
loader_repro = DataLoader(
    windows,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # defina >0 caso o dataset leia blocos do disco a cada __getitem__
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    generator=torch.Generator().manual_seed(SEED),
)


# Funcao para capturar os indices das janelas presentes no primeiro lote
def _first_batch_indices(loader):
    """Retorna os indices de inicio (i_start) das janelas do primeiro lote como lista."""
    _, _, idx_batch = next(iter(loader))
    arr = np.asarray(idx_batch)
    if arr.ndim >= 2:
        return arr[:, 1].tolist()  # coluna i_start_in_trial
    return list(arr)


# Compara a primeira amostragem de duas instancias identicas para verificar determinismo
first_idx_a = _first_batch_indices(loader_repro)
first_idx_b = _first_batch_indices(
    DataLoader(
        windows,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(SEED),
    )
)
# Exibe confirmacao de correspondencia exata entre execucoes
pd.Series(
    {
        "first batch indices (run A)": str(first_idx_a),
        "first batch indices (run B)": str(first_idx_b),
        "match?": first_idx_a == first_idx_b,
    },
    name="value",
).to_frame()

## Etapa 5: Otimizar a velocidade de acesso aleatorio com Zarr

O parametro ``preload=True`` mantem as janelas diretamente na RAM, o que atende perfeitamente a tutoriais e pequenos testes. Em cenarios reais de treinamento, conjuntos de dados raramente cabem por completo na memoria, e arquivos ``.fif`` tornam-se o gargalo (formato sequencial, uma gravacao por arquivo, ausencia de acesso aleatorio em blocos). A solucao do Braindecode e o armazenamento em Zarr: blocos de tamanho fixo, compressao Blosc e um metodo ``__getitem__`` que le qualquer janela em dezenas de milissegundos mesmo quando o dataset atinge centenas de gigabytes.

Essa conversao utiliza o mesmo fluxo executado internamente por :meth:`~braindecode.datasets.BaseConcatDataset.push_to_hub` antes do envio ao Hub; aqui podemos usa-lo offline. Execute essa rotina uma unica vez por projeto: recarregamentos posteriores pagam apenas o custo de descompressao do bloco, e nao o reprocessamento integral. A releitura preserva a mesma interface ``BaseConcatDataset``, portanto o codigo do DataLoader permanece identico.

<div class="alert alert-info"><h4>Nota</h4><p>O Zarr e uma dependencia opcional do Braindecode. Para utiliza-lo, instale:

```bash
pip install braindecode[hub]
```
</p></div>

```python
# Converte janelas para cache baseado em Zarr (uma unica vez, escala linearmente com o tamanho dos dados):
from braindecode.datasets import BaseConcatDataset

zarr_dir = CACHE_DIR / "ds002718_windows.zarr"
windows.push_to_hub(
    repo_id="local-only/ds002718-windows",
    local_cache_dir=zarr_dir,
    compression="blosc",   # blosc oferece melhor desempenho para leitura aleatoria
    compression_level=5,    # nivel de compressao balanceado
    chunk_size=5_000_000,   # amostras por bloco; um bloco = uma operacao de leitura
    token=None,
)

# Recarrega como BaseConcatDataset; a API do DataLoader permanece identica.
windows_zarr = BaseConcatDataset.pull_from_hub(
    repo_id="local-only/ds002718-windows",
    cache_dir=zarr_dir,
    preload=False,            # leituras sob demanda dos blocos
)
```



## Um erro comum e como se recuperar
**Execute.** Solicitar janelas maiores do que a duracao total da gravacao retorna silenciosamente zero janelas. Disparamos isso propositalmente para tornar o modo de falha evidente :cite:`nederbragt2020teaching`: um ``DataLoader(vazio)`` nao gera amostras e mascara o problema na iteracao.



In [ ]:
# Define tamanho de janela excessivo (10 vezes o comprimento total da gravacao)
huge = int(raw_pp.times[-1] * TARGET_SFREQ * 10)  # 10x o comprimento da gravacao
try:
    # Tenta criar janelas com tamanho maior do que a gravacao
    bad = create_fixed_length_windows(
        dataset,
        window_size_samples=huge,
        window_stride_samples=huge,
        drop_last_window=True,
        preload=True,
    )
    print(f"Oversize window produced len={len(bad)} (expected 0).")
except (ValueError, RuntimeError) as exc:
    print(f"Caught {type(exc).__name__}: {str(exc)[:120]}")
# Mensagem de correcao
print(f"Recovery: keep window_size_samples={window_size_samples} (<< recording).")

## Modifique
**Sua vez.** Defina ``WINDOW_SECONDS = 4.0`` e execute novamente a Etapa 3 e a Etapa 4. Preveja antes de rodar: como ``windows[0][0].shape[1]`` deve mudar? Como ``len(windows)`` deve se alterar?



## Crie
**Mini-projeto.** Escreva uma funcao ``collate_fn`` customizada que retorne um *dicionario* em vez da tupla padrao. Lotes em formato de dicionario mantem a legibilidade a medida que o pipeline cresce: HuggingFace ``Trainer``, PyTorch Lightning ``LightningDataModule`` e qualquer chamada do tipo ``model(**batch)`` esperam entradas nomeadas por palavras-chave, documentando explicitamente o papel de cada tensor.



In [ ]:
# Funcao de agrupamento personalizada que monta um dicionario estruturado para cada lote
def dict_collate(batch):
    """Empilha uma lista de tuplas (X, y, idx) em um dicionario de tensores nomeados."""
    return {
        "signal": torch.stack(
            [torch.as_tensor(item[0], dtype=torch.float32) for item in batch], dim=0
        ),
        "target": torch.as_tensor([item[1] for item in batch]),
        "index": torch.as_tensor(
            [
                int(item[2][0])
                if isinstance(item[2], (list, tuple, np.ndarray))
                else int(item[2])
                for item in batch
            ]
        ),
    }


# Instancia DataLoader com a funcao de agrupamento personalizada (collate_fn)
loader_dict = DataLoader(
    windows,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=dict_collate,
)
# Extrai um lote de exemplo para verificar os tipos e dimensoes
batch = next(iter(loader_dict))
pd.Series(
    {
        "type(batch)": type(batch).__name__,
        "keys": str(sorted(batch.keys())),
        "signal.shape": str(tuple(batch["signal"].shape)),
        "signal.dtype": str(batch["signal"].dtype),
        "target.shape": str(tuple(batch["target"].shape)),
        "index.shape": str(tuple(batch["index"].shape)),
    },
    name="value",
).to_frame()

**Investigue.** ``model(**batch)`` agora repassa ``signal=...``, ``target=...``, ``index=...`` diretamente como argumentos nomeados, correspondendo as convencoes modernas do HuggingFace e PyTorch Lightning sem codigo intermediario.



## Janelas continuas vs. epocas baseadas em eventos
Por baixo de cada :class:`~braindecode.datasets.WindowsDataset` reside um objeto :class:`mne.Epochs` (ou :class:`mne.EpochsArray`): um array tridimensional de formato ``(n_epochs, n_channels, n_times)`` juntamente com um dicionario ``event_id`` que mapeia nomes de condicoes para codigos inteiros (como ``{'face': 1, 'scrambled': 2}``) e uma tabela opcional ``metadata`` em :class:`pandas.DataFrame` para filtros no nivel do ensaio. O objeto ``Raw`` representa um sinal continuo; o ``Epochs`` ja e *segmentado*, com seus proprios mecanismos de rejeicao e metadados :cite:`gramfort2013mne`.

Duas funcoes realizam essa transicao a partir do ``Raw``; o DataLoader opera de maneira identica independentemente de qual tenha sido usada.

- :func:`~braindecode.preprocessing.create_fixed_length_windows`: fatia o sinal continuo e atribui a cada janela o mesmo alvo derivado da descricao. Ideal para pre-treinamento autosupervisionado, classificacao de estagios de sono e tarefas continuas.
- :func:`~braindecode.preprocessing.create_windows_from_events`: le os marcadores em :attr:`mne.io.Raw.annotations` (o arquivo ``events.tsv`` e carregado automaticamente pelo EEGDash) e extrai uma janela ao redor de cada evento com os deslocamentos definidos. Ideal para ERPs e decodificacao baseada em eventos (faces vs. imagens embaralhadas em ``ds002718``, P300, imaginacao motora). Use ``mapping={'face': 0, 'scrambled': 1}`` para renomear rotulos diretamente no momento do janelamento.



## Alterar o rotulo sem reconstruir as janelas
O alvo em :class:`~braindecode.datasets.WindowsDataset` reside na coluna ``"target"`` da tabela :class:`pandas.DataFrame` associada aos metadados. O metodo ``__getitem__`` consulta ``self.y[index]``, preenchido a partir dessa coluna. Tres abordagens cobrem os casos praticos:

**Padrao 0: usar um campo BIDS como alvo.** O EEGDash anexa entidades BIDS (``subject``, ``task``, ``session``, ``run``, ``age``, ``gender``, etc.) a propriedade :attr:`~braindecode.datasets.BaseDataset.description`. O janelamento consolida esses dados em :meth:`braindecode.datasets.BaseConcatDataset.get_metadata` (uma linha por janela), permitindo selecionar qualquer coluna como vetor alvo.

**Padrao 1: re-rotular diretamente na memoria.** Altere a coluna de metadados e a lista paralela ``y`` nos subdatasets. A alteracao persiste durante a sessao Python sem necessidade de reprocessar o sinal.

**Padrao 2: mapear na criacao** (janelas por eventos). Passe ``mapping={'face': 0, 'scrambled': 1}`` para :func:`~braindecode.preprocessing.create_windows_from_events`. Os indices numericos sao gravados diretamente em ``metadata['target']``.



Padrao 0: ler campos BIDS como alvo sem modificar os objetos subjacentes.



In [ ]:
# Extrai a tabela consolidada de metadados das janelas
metadata = windows.get_metadata()
# Seleciona a coluna de sujeitos como alvo
y_subject = metadata["subject"].to_numpy()
# Exibe resumo da extracao de alvos
pd.Series(
    {
        "rows in metadata": len(metadata),
        "task unique": str(metadata["task"].unique().tolist()),
        "subject unique": str(metadata["subject"].unique().tolist()),
        "y_subject dtype": str(y_subject.dtype),
        "windows.datasets[0].description['task']": str(
            windows.datasets[0].description.get("task")
        ),
    },
    name="value",
).to_frame()

## Padrao 1: re-rotular na memoria
Adequado quando o rotulo decorre de uma computacao dinamica: tarefas autosupervisionadas, classificacao deslizante ou saidas de modelos pre-treinados.



In [ ]:
# Seleciona o primeiro subconjunto de janelas
sub_ds = windows.datasets[0]
n = len(sub_ds)
half = n // 2
# Define artificialmente rotulo 0 para a primeira metade e rotulo 1 para a segunda
new_targets = [0] * half + [1] * (n - half)
# Atualiza simultaneamente o atributo .y e a coluna target do DataFrame de metadados
sub_ds.y = new_targets
sub_ds.metadata.loc[:, "target"] = new_targets

# Recarrega o DataLoader para verificar os novos alvos atribuidos
loader_relabelled = DataLoader(
    windows, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
_, y_relabelled, _ = next(iter(loader_relabelled))
# Exibe confirmacao da alteracao dos rotulos
pd.Series(
    {
        "before (orig)": "all windows shared the recording-level target",
        "after y": str(torch.unique(y_relabelled).tolist()),
        "first 8 windows": str(new_targets[:8]),
    },
    name="value",
).to_frame()

## Resultado
Convertemos um sujeito de ``ds002718`` em um ``DataLoader`` reprodutivel do PyTorch: consulta BIDS, dois pre-processadores seguros, janelamento de comprimento fixo e leitura em lotes. O primeiro lote possui formato ``(batch_size, n_channels, window_samples)`` com tipo float32.



## Conclusao
Partimos de um :class:`~eegdash.api.EEGDashDataset` e chegamos a um DataLoader pronto para o modelo. A seguir: :doc:`/generated/auto_examples/tutorials/10_core_workflow/plot_10_preprocess_and_window` aborda a receita completa de pre-processamento; :doc:`/generated/auto_examples/tutorials/10_core_workflow/plot_11_leakage_safe_split` divide os dados sem vazamento entre sujeitos; e :doc:`/generated/auto_examples/tutorials/10_core_workflow/plot_13_save_and_reuse_prepared_data` salva e reutiliza janelas prontas.



## Tente voce mesmo
- Reexecute com ``shuffle=True`` e ``generator=None``: observe que a ordem do primeiro lote torna-se variavel entre execucoes. Restaure o ``Generator`` deterministico e confirme a reprodutibilidade.
- Configure ``num_workers=2`` e confirme que a saida do lote permanece identica.
- Substitua ``window_stride_samples = window_size_samples`` por uma sobreposicao de 50% (``window_size_samples // 2``) e preveja a nova contagem total de janelas.



## Referencias
Consulte :doc:`/references` para a bibliografia centralizada dos artigos citados acima.

